# 09 · DPO on the SFT model's own mistakes (`dpo_v2`, `dpo_v3`)

**Why a new DPO.** Our first DPO model (`dpo_v1`, notebook 05) made the SFT model worse (puzzle accuracy 7.8% → 5.9%). Two things in its setup are likely causes:
1. **Its rejected moves were too easy.** Each rejected move was a *random* legal move losing at least 200 centipawns (median: 620, about six pawns). These are moves the model would rarely play, so learning to avoid them teaches little. Our TA suggested changing how the pairs are built.
2. **It lowered the probability of the good move too.** DPO only asks the chosen move to gain *relative to* the rejected one. In `dpo_v1` both lost probability, and the freed probability went partly to illegal moves.

**What this notebook does.** We first run the SFT model (`sft_v1`) on training positions it has never seen and collect its **own mistakes**: answers that are illegal, or legal but losing at least 200 centipawns. Each mistake becomes the *rejected* answer of a pair, with Stockfish's best move as the *chosen* answer. Then two DPO runs, each starting from `sft_v1`:

| | `dpo_v1` (notebook 05) | `dpo_v2` | `dpo_v3` |
|---|---|---|---|
| Chosen answer | Stockfish's best move | the same | the same |
| Rejected answer | a random move losing 200+ cp | **the SFT model's own mistake** (illegal, or losing 200+ cp) | the same as `dpo_v2` |
| Loss | DPO | DPO | **DPO + SFT loss on the chosen move** |
| Number of pairs | 4,788 (4,549 for training) | the same number | the same pairs |
| Everything else (β = 0.1, learning rate, epochs, LoRA) | `configs/dpo.yaml` | the same | the same |

So `dpo_v2` vs. `dpo_v1` measures the effect of the rejected moves, and `dpo_v3` vs. `dpo_v2` the effect of the added SFT loss.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run.
3. Set `QUICK_CHECK = False` and run the real thing. For a run this long, use a **Job** so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification.
4. Afterwards, commit `results/` from the Git folder; Databricks cannot commit notebook outputs. Then **File → Export → IPython Notebook**, replace this notebook with the exported file on a laptop, and commit it there, so the figures also show inside the notebook on GitHub.

**Time:** about 45–60 minutes to collect the mistakes (the SFT model answers positions on the GPU, Stockfish checks them), then about 20 minutes of training and 5 minutes of grading for each of the two runs.

**Outputs**
- the pairs in `data/processed/dpo_pairs_sft_mistakes.jsonl` (not on GitHub; reused if the notebook runs again)
- the adapters in `checkpoints/dpo_v2/` and `checkpoints/dpo_v3/` (not on GitHub)
- results: `results/dpo_v2*.json`, `results/dpo_v3*.json`, and new rows in `results/summary.csv`
- figures and tables: `results/figures/dpo_mistake_pairs/` (the pairs), `results/figures/dpo_v2/`, `results/figures/dpo_v3/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
QUICK_CHECK = True    # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
RUNS = {              # run name -> its change to the DPO training settings
    "dpo_v2": {"loss_type": ["sigmoid"]},          # the standard DPO loss, as in dpo_v1
    "dpo_v3": {"loss_type": ["sigmoid", "sft"]},   # plus the SFT loss on the chosen move
}

import copy
import gc

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import (compare_with, figures_dir, grade_puzzles, puzzle_table, reply_function,
                                 save_table, save_training)
from chesslm.preferences import build_mistake_pairs
from chesslm.train_dpo import split_pairs, train
import matplotlib.pyplot as plt

sft_cfg, prompt_cfg, eval_cfg = load_config("sft"), load_config("prompt"), load_config("eval")
dpo_cfg = load_config("dpo")
dpo_cfg["rejected_source"] = "sft_mistakes"   # THE change in the pairs
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
SFT_ADAPTER = REPO_ROOT / dpo_cfg["training"]["start_adapter"]   # sft_v1: makes the mistakes, and DPO starts from it
PAIRS_FILE = REPO_ROOT / "data" / "processed" / "dpo_pairs_sft_mistakes.jsonl"
PAIRS_FIGURES = "dpo_mistake_pairs"

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUNS = {f"{name}_check": change for name, change in RUNS.items()}
    PAIRS_FILE = PAIRS_FILE.with_name("dpo_pairs_quick_check.jsonl")
    PAIRS_FIGURES += "_check"
    dpo_cfg["target_pairs"] = 24
    dpo_cfg["training"].update(logging_steps=1, eval_steps=1, batch_size=4, effective_batch_size=8,
                               val_fraction=0.25)
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
print(f"Runs: {list(RUNS)} | pairs file: {PAIRS_FILE.name} | SFT model: {SFT_ADAPTER}")

## 1. Collect the SFT model's mistakes

**Which positions.** Training positions from strong players' games, shuffled with a fixed seed. We leave out the 20,000 positions `sft_v1` was trained on: its answers there may be memorized, and we want the mistakes it makes on new positions. Puzzles are never used.

**How.** `sft_v1` answers the positions in batches of 2,000, with the same prompt and greedy decoding as in grading. Stockfish checks each answer. Every illegal or unparseable answer, and every legal answer losing at least 200 centipawns, becomes a pair. We stop at 4,788 pairs, the same number as `dpo_v1`, so both DPO runs see the same amount of data.

In [ ]:
from chesslm.train_sft import sample_examples

train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
sft_positions = sample_examples(train_positions, sft_cfg["data"]["train_examples"], sft_cfg["training"]["seed"])
candidates = train_positions.drop(sft_positions.index).sample(frac=1, random_state=dpo_cfg["seed"])
print(f"{len(candidates):,} candidate positions (training positions minus the {len(sft_positions):,} sft_v1 trained on)")

if PAIRS_FILE.exists():
    pairs = pd.read_json(PAIRS_FILE, lines=True)
    print(f"Loaded {len(pairs):,} pairs from {PAIRS_FILE.name}")
else:
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=SFT_ADAPTER)
    reply = reply_function(model, tokenizer, prompt_cfg, eval_cfg["generation"])
    pairs, answered = build_mistake_pairs(candidates, reply, prompt_cfg, dpo_cfg,
                                          {**eval_cfg["stockfish"], "depth": dpo_cfg["stockfish_depth"]},
                                          batch_positions=64 if QUICK_CHECK else 2000)
    pairs["positions_answered"] = answered
    pairs.to_json(PAIRS_FILE, orient="records", lines=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()
answered = int(pairs.positions_answered.iloc[0])

**What the mistakes look like.** The table compares the new pairs with `dpo_v1`'s. "Mistake rate" is the share of answered positions where `sft_v1` made a mistake. For legal rejected moves, the centipawn loss shows how bad they are: `dpo_v1`'s random moves are mostly severe blunders, while the model's own mistakes may be closer calls.

In [ ]:
FIG = figures_dir(PAIRS_FIGURES)
kinds = pairs.rejected_kind.value_counts(normalize=True) * 100
legal = pairs.rejected_cp_loss.dropna()
rows = {"dpo_v2 / dpo_v3 (the model's own mistakes)": {
    "Pairs": len(pairs), "Positions answered": answered, "Mistake rate (%)": round(len(pairs) / answered * 100, 1),
    "Rejected: illegal (%)": round(kinds.get("illegal", 0), 1),
    "Rejected: unparseable (%)": round(kinds.get("unparseable", 0), 1),
    "Rejected: legal, loses 200+ cp (%)": round(kinds.get("blunder", 0), 1),
    "Median loss of legal rejected moves (cp)": legal.median() if len(legal) else None}}
v1_file = REPO_ROOT / "data" / "processed" / "dpo_pairs.jsonl"
if v1_file.exists():
    v1 = pd.read_json(v1_file, lines=True)
    rows["dpo_v1 (random moves)"] = {"Pairs": len(v1), "Positions answered": None, "Mistake rate (%)": None,
        "Rejected: illegal (%)": 0.0, "Rejected: unparseable (%)": 0.0, "Rejected: legal, loses 200+ cp (%)": 100.0,
        "Median loss of legal rejected moves (cp)": v1.rejected_cp_loss.median()}
pair_table = pd.DataFrame(rows).T
pair_table.index.name = "Pairs"
save_table(pair_table, FIG, "pairs", "Rejected answers in the preference pairs. cp = centipawns (100 = one pawn).")
display(pair_table)

if len(legal):
    plots.histogram(legal, "centipawns lost by the rejected (legal) move", FIG / "rejected_move_loss.png",
                    edges=[200], title="How bad are the model's legal mistakes? (dashed line: the 200 cp threshold)");

print("\nThree pairs, as DPO sees them:")
for row in pairs.head(3).itertuples():
    print(row.prompt[0]["content"])
    print(f"  chosen:   {row.chosen[0]['content']}")
    print(f"  rejected: {row.rejected[0]['content']}  ({row.rejected_kind})\n")

## 2. Train and grade `dpo_v2` and `dpo_v3`

Both runs use the same pairs and the same 5% of pairs held out for monitoring, start from `sft_v1`, and differ only in the loss. For each run, two figures:
- **training loss**
- **DPO diagnostics.** Left: how often the model now prefers the chosen answer over the rejected one (50% = no preference; it should rise). Right: the probability of the chosen and rejected answers compared with the SFT model (1× = unchanged). In `dpo_v1` both fell; we hope the chosen line now stays near or above 1×, especially in `dpo_v3`.

Each model is then graded on the dev and test puzzles exactly like every other model.

In [ ]:
train_ds, val_ds = split_pairs(pairs, dpo_cfg["training"]["val_fraction"], dpo_cfg["training"]["seed"])
print(f"train: {len(train_ds):,} pairs  held out: {len(val_ds):,} pairs\n")

for run_name, change in RUNS.items():
    cfg = copy.deepcopy(dpo_cfg)
    cfg["training"].update(change)
    adapter_dir = REPO_ROOT / "checkpoints" / run_name
    fig = figures_dir(run_name)
    print(f"=== {run_name}: loss {cfg['training']['loss_type']} ===")

    log, compute = train(train_ds, val_ds, cfg, sft_cfg["model"], sft_cfg["lora"], SFT_ADAPTER, adapter_dir)
    save_training(run_name, {"dpo": cfg, "prompt": prompt_cfg}, compute, log, start_adapter=SFT_ADAPTER,
                  extra={"pairs_file": PAIRS_FILE.name, "pairs": len(pairs)})
    plots.loss_curve(log, fig / "training_loss.png", title=f"{run_name}: loss during training")
    plots.dpo_diagnostics(log, cfg["training"]["beta"], fig / "dpo_diagnostics.png",
                          title=f"{run_name}: does the model prefer Stockfish's move over its own mistake?",
                          rejected_label="rejected (the model's own mistake)")
    plt.show()

    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter_dir)
    grade_puzzles(model, tokenizer, "dev", f"{run_name}_dev", configs, str(adapter_dir), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", run_name, configs, str(adapter_dir), PUZZLE_LIMIT)
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 3. Results: SFT vs. the three DPO models

**How to read this section**
- **Puzzle accuracy** is the main metric (a random legal move solves 5.6%). DPO starts from `sft_v1`, so `sft_v1` is the line to beat.
- **Legal / illegal / unparseable:** `dpo_v1` raised the illegal rate (28% → 37%). Pairs whose rejected answer is an illegal move should push this back down.
- **ACPL:** how much worse the legal answers are than Stockfish's best move (centipawns; lower is better).
- The **paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance.

All numbers are on the 1,000 test puzzles. These figures and tables are saved in the folder of each new run.

In [ ]:
v2, v3 = RUNS
runs = ["sft_v1", "dpo_v1", v2, v3]
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
comparison = pd.DataFrame([compare_with(v2, "sft_v1"), compare_with(v2, "dpo_v1"),
                           compare_with(v3, "sft_v1"), compare_with(v3, v2)]).set_index("model")

for run_name in RUNS:  # the same comparison is saved in each new run's folder
    fig = figures_dir(run_name)
    table = puzzle_table(runs, fig)
    save_table(comparison, fig, "paired_comparison",
               "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
    plt.close("all")  # only the last copy of each figure is shown below
    plots.compare_metrics(summary, runs, fig / "puzzle_metrics.png", title="SFT and the three DPO models")
    plots.accuracy_by_band(summary, runs, fig / "accuracy_by_band.png", title="Puzzle accuracy by puzzle difficulty")
    plots.outcome_breakdown(summary, runs, fig / "answer_outcomes.png", title="Are the answers legal moves?")

display(table)
display(comparison)

## Summary of outputs

- `results/figures/dpo_mistake_pairs/`: `pairs` (table: what the rejected answers are), `rejected_move_loss.png`
- `results/figures/dpo_v2/` and `results/figures/dpo_v3/`: `training_loss.png`, `dpo_diagnostics.png`, `puzzle_results` (table), `paired_comparison` (table), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/dpo_v2.json`, `results/dpo_v3.json`: every test answer and its grade; `*_training.json`: settings, compute, training log
- Both models are also graded on held-out game positions and in full games in notebook 11.